In [1]:
import io
import time
import requests
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

# 1. Log in with your Google Account
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# ----------------- SETTINGS -----------------
SPREADSHEET_ID = '1fuPsAOQYF3AHqGSm2nB84NKWyoTkms8FHz1O6eymOwE'   # Get from your sheet's URL
SHEET_NAME = 'Inv X L30 sales'                # Tab name
CSV_URL = 'https://watchtower.partnr.in/public/question/d955c6bb-6062-41d8-93df-f9176afd4390.csv'
CHUNK_SIZE = 25000                           # 10k rows per batch = only ~33 writes total
THROTTLE_DELAY = 2.5
# --------------------------------------------

# Open sheet and worksheet
sh = gc.open_by_key(SPREADSHEET_ID)
ws = sh.worksheet(SHEET_NAME)

# Step A: Clear Columns A to D (preserving Row 1 header)
print("Clearing columns A:D...")
ws.batch_clear(['A2:D'])
print("Columns A:D cleared.")

# Step B: Download CSV
print("Downloading CSV...")
response = requests.get(CSV_URL)
response.raise_for_status()

# Step C: Read first 4 columns
print("Parsing data...")
df = pd.read_csv(
    io.StringIO(response.text),
    usecols=[0, 1, 2, 3],
    dtype=str
).fillna('')

total_rows = len(df)
print(f"Total rows to write: {total_rows}")

# Step D: Upload in batches
current_row = 2
for i in range(0, total_rows, CHUNK_SIZE):
    chunk = df.iloc[i : i + CHUNK_SIZE].values.tolist()
    end_row = current_row + len(chunk) - 1
    range_label = f"A{current_row}:D{end_row}"

    print(f"Writing rows {current_row} to {end_row}...")
    ws.update(range_name=range_label, values=chunk, value_input_option='USER_ENTERED')

    current_row += len(chunk)
    time.sleep(1.5)  # Safe pause to stay within Google limits

print("Done! All data imported successfully.")

Clearing columns A:D...
Columns A:D cleared.
Parsing data...
Total rows to write: 334570
Writing rows 2 to 25001...
Writing rows 25002 to 50001...
Writing rows 50002 to 75001...
Writing rows 75002 to 100001...
Writing rows 100002 to 125001...
Writing rows 125002 to 150001...
Writing rows 150002 to 175001...
Writing rows 175002 to 200001...
Writing rows 200002 to 225001...
Writing rows 225002 to 250001...
Writing rows 250002 to 275001...
Writing rows 275002 to 300001...
Writing rows 300002 to 325001...
Writing rows 325002 to 334571...
Done! All data imported successfully.
